# Raw vs Cleaned Quality Check

This brief notebook follows the evaluation framing in Section 3 of the README: compare the raw dataset with the cleaned output using measurable quality signals and the pipeline's own verification artifacts. It does not call agents; it reads the cached validation, cleaning, and verification results produced by the pipeline.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import altair as alt

DATASET = "attivazioniCessazioni"

RAW_PATH = Path("Data") / f"{DATASET}.csv"
CLEANED_PATH = Path("Data/.cleaning_cache") / DATASET / f"{DATASET}.cleaned.csv"
VALIDATION_BUNDLE_PATH = Path("Data/.validation_cache") / f"{DATASET}.validation_bundle.json"
FINAL_REPORT_PATH = Path("Data/.cleaning_cache") / DATASET / f"{DATASET}.final_report.json"
CLEANER_MANIFEST_PATH = Path("Data/.cleaning_cache") / DATASET / "cleaner_manifest.json"

raw = pd.read_csv(RAW_PATH, dtype="string")
cleaned = pd.read_csv(CLEANED_PATH, dtype="string")
validation_bundle = json.loads(VALIDATION_BUNDLE_PATH.read_text())
final_report = json.loads(FINAL_REPORT_PATH.read_text())
cleaner_manifest = json.loads(CLEANER_MANIFEST_PATH.read_text())

print(f"Raw shape:     {raw.shape}")
print(f"Cleaned shape: {cleaned.shape}")

Raw shape:     (20102, 19)
Cleaned shape: (20077, 19)


## Table-Level Quality Signals

These are deterministic checks that can be computed directly on the raw and cleaned CSVs: shape, missing-like cells, exact duplicate rows after simple normalization, and unsafe column names.

In [2]:
PLACEHOLDER_TOKENS = {"", "na", "n/a", "null", "none", "-", "--", "unknown", "n.d.", "?", "//", "nan"}
VALID_SCHEMA_NAME_RE = re.compile(r"^[a-z][a-z0-9_]*$")


def missing_like_mask(df: pd.DataFrame) -> pd.DataFrame:
    rendered = df.astype("string").apply(lambda col: col.str.strip().str.lower())
    return df.isna() | rendered.isin(PLACEHOLDER_TOKENS)


def normalized_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df.astype("string").fillna("").apply(lambda col: col.str.strip().str.lower())


def table_quality_profile(label: str, df: pd.DataFrame) -> dict[str, float | int | str]:
    missing_mask = missing_like_mask(df)
    total_cells = int(df.shape[0] * df.shape[1])
    missing_like_cells = int(missing_mask.to_numpy().sum())
    duplicate_rows = int(normalized_rows(df).duplicated(keep="first").sum())
    unsafe_columns = sum(not VALID_SCHEMA_NAME_RE.fullmatch(column) for column in df.columns)
    return {
        "dataset": label,
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "cells": total_cells,
        "missing_like_cells": missing_like_cells,
        "missing_like_pct": round(missing_like_cells / total_cells * 100, 2) if total_cells else 0.0,
        "complete_cell_pct": round((total_cells - missing_like_cells) / total_cells * 100, 2) if total_cells else 0.0,
        "normalized_exact_duplicate_rows": duplicate_rows,
        "unsafe_column_names": int(unsafe_columns),
    }


quality = pd.DataFrame([
    table_quality_profile("raw", raw),
    table_quality_profile("cleaned", cleaned),
]).set_index("dataset")

delta = quality.loc["cleaned"] - quality.loc["raw"]
delta.name = "delta_cleaned_minus_raw"
display(pd.concat([quality, delta.to_frame().T]))

,rows,columns,cells,missing_like_cells,missing_like_pct,complete_cell_pct,normalized_exact_duplicate_rows,unsafe_column_names
raw,20102.0,19.0,381938.0,49405.0,12.94,87.06,67.0,8.0
cleaned,20077.0,19.0,381463.0,49910.0,13.08,86.92,42.0,1.0
delta_cleaned_minus_raw,-25.0,0.0,-475.0,505.0,0.14,-0.14,-25.0,-7.0


## Pipeline Finding Counts

Section 3 describes evaluation through findings, actions, accepted cleaners, and verification outcomes. The next tables collect those measurements from the cached validation bundle and final report.

In [3]:
raw_findings = pd.Series({
    "schema_issues": len(validation_bundle["schema_validation"].get("issues", [])),
    "columns_with_missing_values": len(validation_bundle["completeness_analysis"].get("columns_with_missing_values", [])),
    "format_consistency_findings": len(validation_bundle["consistency_validation"].get("format_consistency_findings", [])),
    "anomaly_findings": len(validation_bundle["anomaly_detection"].get("findings", [])),
    "cross_column_findings": len(validation_bundle["cross_column_validation"].get("findings", [])),
    "duplicate_groups": len(validation_bundle["duplicate_detection"].get("groups", [])),
}, name="raw_validation_count")

action_summary = pd.Series({
    "applied_actions": len(final_report.get("applied_actions", [])),
    "proposed_not_applied_actions": len(final_report.get("proposed_not_applied_actions", [])),
    "failed_actions": len(final_report.get("failed_actions", [])),
    "generated_cleaners_accepted": len(cleaner_manifest),
    "total_rows_cleaned": final_report.get("total_rows_cleaned"),
}, name="cleaning_process_count")

display(raw_findings.to_frame())
display(action_summary.to_frame())

,raw_validation_count
schema_issues,13
columns_with_missing_values,11
format_consistency_findings,6
anomaly_findings,0
cross_column_findings,7
duplicate_groups,48


,cleaning_process_count
applied_actions,59
proposed_not_applied_actions,30
failed_actions,0
generated_cleaners_accepted,6
total_rows_cleaned,20077


## Verification Outcomes

The strongest before-versus-after check is the verification diff: each targeted format issue is re-measured on the cleaned output and classified as resolved, improved, unchanged, or regressed.

In [4]:
verification = pd.DataFrame(final_report.get("verification_diffs", []))

if verification.empty:
    print("No verification diffs found in the final report.")
else:
    verification_view = verification[[
        "column_name",
        "renamed_to",
        "status",
        "before_inconsistent_rows",
        "after_inconsistent_rows",
        "reduction_pct",
    ]].copy()
    verification_totals = pd.Series({
        "targeted_columns": len(verification),
        "before_inconsistent_rows": int(verification["before_inconsistent_rows"].sum()),
        "after_inconsistent_rows": int(verification["after_inconsistent_rows"].sum()),
        "resolved": int((verification["status"] == "resolved").sum()),
        "improved": int((verification["status"] == "improved").sum()),
        "unchanged": int((verification["status"] == "unchanged").sum()),
        "regressed": int((verification["status"] == "regressed").sum()),
    }, name="verification_totals")
    verification_totals["overall_inconsistent_row_reduction_pct"] = round(
        (1 - verification_totals["after_inconsistent_rows"] / verification_totals["before_inconsistent_rows"]) * 100,
        2,
    ) if verification_totals["before_inconsistent_rows"] else 0.0

    display(verification_view)
    display(verification_totals.to_frame())

,column_name,renamed_to,status,before_inconsistent_rows,after_inconsistent_rows,reduction_pct
0,mese,NaN,resolved,571,0,100.0
1,anno,NaN,resolved,585,0,100.0
2,attivazioni,NaN,resolved,414,0,100.0
3,cessazioni,NaN,resolved,413,0,100.0
4,RATA,rata,resolved,802,0,100.0
5,aggregation-time,aggregation_time,resolved,1607,0,100.0


,verification_totals
targeted_columns,6.0
before_inconsistent_rows,4392.0
after_inconsistent_rows,0.0
resolved,6.0
improved,0.0
unchanged,0.0
regressed,0.0
overall_inconsistent_row_reduction_pct,100.0


## Cleaner Impact

Accepted cleaners are summarized below with changed-row counts. These counts indicate where executable cleaning actually changed values; preserved examples in the manifest are evidence that already-valid values were checked during host-side verification.

In [5]:
cleaners = pd.DataFrame(cleaner_manifest)

if cleaners.empty:
    print("No generated cleaners were recorded.")
else:
    cleaner_view = cleaners[["column_name", "function_name", "changed_rows", "summary"]].copy()
    cleaner_totals = pd.Series({
        "accepted_cleaners": len(cleaners),
        "total_changed_rows_across_cleaners": int(cleaners["changed_rows"].sum()),
        "median_changed_rows_per_cleaner": float(cleaners["changed_rows"].median()),
    }, name="cleaner_totals")

    display(cleaner_view.sort_values("changed_rows", ascending=False))
    display(cleaner_totals.to_frame())

,column_name,function_name,changed_rows,summary
5,aggregation-time,clean_aggregation_time,1607,Applied successfully: 1607 rows changed.
4,RATA,clean_RATA,802,Applied successfully: 802 rows changed.
0,mese,clean_mese,744,Applied successfully: 744 rows changed.
2,attivazioni,clean_COLUMN,609,Applied successfully: 609 rows changed.
3,cessazioni,clean_COLUMN,608,Applied successfully: 608 rows changed.
1,anno,clean_COLUMN,585,Applied successfully: 585 rows changed.


,cleaner_totals
accepted_cleaners,6.0
total_changed_rows_across_cleaners,4955.0
median_changed_rows_per_cleaner,676.5


## Visual Results for the README

These charts are intentionally simple: each one answers a single before-versus-after question that can be read without knowing the internals of the pipeline.

In [6]:
quality_chart_data = (
    quality.reset_index()[[
        "dataset",
        "missing_like_cells",
        "normalized_exact_duplicate_rows",
        "unsafe_column_names",
    ]]
    .melt(id_vars="dataset", var_name="metric", value_name="count")
)
quality_chart_data["metric"] = quality_chart_data["metric"].map({
    "missing_like_cells": "Missing-like cells",
    "normalized_exact_duplicate_rows": "Exact duplicate rows",
    "unsafe_column_names": "Unsafe column names",
})

alt.Chart(quality_chart_data).mark_bar().encode(
    x=alt.X("dataset:N", title=None),
    y=alt.Y("count:Q", title="Count"),
    color=alt.Color("dataset:N", title="Dataset"),
    column=alt.Column("metric:N", title=None),
    tooltip=["metric", "dataset", alt.Tooltip("count:Q", format=",")],
).properties(
    title="Raw vs cleaned quality signals",
    width=150,
    height=220,
)


alt.Chart(...)

In [7]:
verification_chart_data = verification_view.rename(columns={
    "before_inconsistent_rows": "Raw",
    "after_inconsistent_rows": "Cleaned",
}).melt(
    id_vars=["column_name", "status"],
    value_vars=["Raw", "Cleaned"],
    var_name="dataset",
    value_name="inconsistent_rows",
)

alt.Chart(verification_chart_data).mark_bar().encode(
    y=alt.Y("column_name:N", sort="-x", title="Targeted column"),
    x=alt.X("inconsistent_rows:Q", title="Inconsistent rows"),
    color=alt.Color("dataset:N", title="Dataset"),
    tooltip=["column_name", "dataset", alt.Tooltip("inconsistent_rows:Q", format=",")],
).properties(
    title="Targeted format issues were eliminated after cleaning",
    width=520,
    height=260,
)


alt.Chart(...)

In [8]:
cleaner_chart_data = cleaner_view.sort_values("changed_rows", ascending=False).copy()

alt.Chart(cleaner_chart_data).mark_bar().encode(
    y=alt.Y("column_name:N", sort="-x", title="Cleaner target"),
    x=alt.X("changed_rows:Q", title="Rows changed"),
    tooltip=["column_name", "function_name", alt.Tooltip("changed_rows:Q", format=",")],
).properties(
    title="Accepted cleaner impact by column",
    width=520,
    height=240,
)


alt.Chart(...)

## Infographic Summary

These cells are designed as screenshot-friendly figures for the README: large numbers first, then one chart for resolution and one chart for remaining caveats.

In [9]:
from IPython.display import HTML

before_inconsistent = int(verification["before_inconsistent_rows"].sum()) if not verification.empty else 0
after_inconsistent = int(verification["after_inconsistent_rows"].sum()) if not verification.empty else 0
resolved = int((verification["status"] == "resolved").sum()) if not verification.empty else 0
dropped_rows = int(raw.shape[0] - cleaned.shape[0])
changed_values = int(cleaners["changed_rows"].sum()) if not cleaners.empty else 0
unsafe_before = int(quality.loc["raw", "unsafe_column_names"])
unsafe_after = int(quality.loc["cleaned", "unsafe_column_names"])
duplicate_before = int(quality.loc["raw", "normalized_exact_duplicate_rows"])
duplicate_after = int(quality.loc["cleaned", "normalized_exact_duplicate_rows"])

HTML(f"""
<div style="font-family: Inter, Arial, sans-serif; border:1px solid #e6e1d8; padding:22px; border-radius:10px; background:#fffaf0; max-width:980px;">
  <div style="font-size:14px; text-transform:uppercase; letter-spacing:.08em; color:#7a5b2e; font-weight:700;">Data Quality Lift</div>
  <div style="font-size:30px; line-height:1.15; font-weight:800; color:#172026; margin:6px 0 18px;">{DATASET}: raw data converted into verified cleaned output</div>
  <div style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;">
    <div style="background:white; border-left:6px solid #227c70; padding:14px; border-radius:8px; box-shadow:0 1px 6px #00000012;">
      <div style="font-size:28px; font-weight:800; color:#172026;">{before_inconsistent:,} → {after_inconsistent:,}</div>
      <div style="font-size:13px; color:#4a5560;">targeted inconsistent rows</div>
    </div>
    <div style="background:white; border-left:6px solid #d1495b; padding:14px; border-radius:8px; box-shadow:0 1px 6px #00000012;">
      <div style="font-size:28px; font-weight:800; color:#172026;">{resolved}/6</div>
      <div style="font-size:13px; color:#4a5560;">format findings resolved</div>
    </div>
    <div style="background:white; border-left:6px solid #f2a541; padding:14px; border-radius:8px; box-shadow:0 1px 6px #00000012;">
      <div style="font-size:28px; font-weight:800; color:#172026;">{changed_values:,}</div>
      <div style="font-size:13px; color:#4a5560;">row values changed by accepted cleaners</div>
    </div>
    <div style="background:white; border-left:6px solid #3d5a80; padding:14px; border-radius:8px; box-shadow:0 1px 6px #00000012;">
      <div style="font-size:28px; font-weight:800; color:#172026;">{dropped_rows}</div>
      <div style="font-size:13px; color:#4a5560;">exact duplicate rows removed</div>
    </div>
  </div>
  <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-top:12px;">
    <div style="background:#172026; color:white; padding:14px; border-radius:8px;">
      <div style="font-size:13px; color:#d7e4e0;">Schema usability</div>
      <div style="font-size:23px; font-weight:800;">Unsafe names: {unsafe_before} → {unsafe_after}</div>
    </div>
    <div style="background:#172026; color:white; padding:14px; border-radius:8px;">
      <div style="font-size:13px; color:#d7e4e0;">Row redundancy</div>
      <div style="font-size:23px; font-weight:800;">Duplicate rows: {duplicate_before} → {duplicate_after}</div>
    </div>
  </div>
</div>
""")


In [10]:
resolution_data = pd.DataFrame({
    "status": ["Resolved", "Remaining"],
    "columns": [resolved, max(len(verification) - resolved, 0)],
})

donut = alt.Chart(resolution_data).mark_arc(innerRadius=58, outerRadius=92).encode(
    theta=alt.Theta("columns:Q"),
    color=alt.Color("status:N", scale=alt.Scale(range=["#227c70", "#e6e1d8"]), legend=None),
    tooltip=["status", "columns"],
).properties(width=240, height=240, title="6 of 6 targeted columns resolved")

donut_text = alt.Chart(pd.DataFrame({"label": [f"{resolved}/{len(verification)}"], "sub": ["resolved"]})).mark_text(
    align="center", baseline="middle", dy=-8, fontSize=34, fontWeight="bold", color="#172026"
).encode(text="label:N") + alt.Chart(pd.DataFrame({"sub": ["resolved"]})).mark_text(
    align="center", baseline="middle", dy=24, fontSize=13, color="#4a5560"
).encode(text="sub:N")

reduction_data = pd.DataFrame([
    {"stage": "Raw", "rows": before_inconsistent},
    {"stage": "Cleaned", "rows": after_inconsistent},
])

bars = alt.Chart(reduction_data).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    y=alt.Y("stage:N", sort=["Raw", "Cleaned"], title=None),
    x=alt.X("rows:Q", title="Targeted inconsistent rows"),
    color=alt.Color("stage:N", scale=alt.Scale(range=["#d1495b", "#227c70"]), legend=None),
    tooltip=["stage", alt.Tooltip("rows:Q", format=",")],
).properties(width=470, height=150, title="Format inconsistency reduction")

labels = alt.Chart(reduction_data).mark_text(align="left", dx=6, fontWeight="bold", color="#172026").encode(
    y=alt.Y("stage:N", sort=["Raw", "Cleaned"]),
    x="rows:Q",
    text=alt.Text("rows:Q", format=","),
)

(donut + donut_text) | (bars + labels)


alt.HConcatChart(...)

In [11]:
risk_data = pd.DataFrame([
    {"category": "Auto-resolved format findings", "count": resolved, "kind": "resolved"},
    {"category": "Cross-column findings still need review", "count": len(final_report.get("cross_column_findings", [])), "kind": "review"},
    {"category": "Duplicate groups still need review", "count": len(final_report.get("duplicate_groups", [])), "kind": "review"},
    {"category": "Anomaly findings", "count": len(final_report.get("anomaly_findings", [])), "kind": "review"},
])

alt.Chart(risk_data).mark_bar().encode(
    y=alt.Y("category:N", sort="-x", title=None),
    x=alt.X("count:Q", title="Count"),
    color=alt.Color("kind:N", scale=alt.Scale(domain=["resolved", "review"], range=["#227c70", "#f2a541"]), title="Interpretation"),
    tooltip=["category", alt.Tooltip("count:Q", format=",")],
).properties(
    title="Clear win, honest caveat: what improved vs what remains for review",
    width=650,
    height=220,
)


alt.Chart(...)

## README-Ready Key Points

The next cell prints compact bullets and a small table that can be copied into the README results section.

In [12]:
before_inconsistent = int(verification["before_inconsistent_rows"].sum()) if not verification.empty else 0
after_inconsistent = int(verification["after_inconsistent_rows"].sum()) if not verification.empty else 0
resolved = int((verification["status"] == "resolved").sum()) if not verification.empty else 0
dropped_rows = int(raw.shape[0] - cleaned.shape[0])

readme_metrics = pd.DataFrame([
    {"metric": "Rows", "raw": raw.shape[0], "cleaned": cleaned.shape[0], "change": cleaned.shape[0] - raw.shape[0]},
    {"metric": "Unsafe column names", "raw": quality.loc["raw", "unsafe_column_names"], "cleaned": quality.loc["cleaned", "unsafe_column_names"], "change": quality.loc["cleaned", "unsafe_column_names"] - quality.loc["raw", "unsafe_column_names"]},
    {"metric": "Normalized exact duplicate rows", "raw": quality.loc["raw", "normalized_exact_duplicate_rows"], "cleaned": quality.loc["cleaned", "normalized_exact_duplicate_rows"], "change": quality.loc["cleaned", "normalized_exact_duplicate_rows"] - quality.loc["raw", "normalized_exact_duplicate_rows"]},
    {"metric": "Targeted inconsistent rows", "raw": before_inconsistent, "cleaned": after_inconsistent, "change": after_inconsistent - before_inconsistent},
    {"metric": "Resolved targeted columns", "raw": 0, "cleaned": resolved, "change": resolved},
])

display(readme_metrics)

print("README bullets:")
print(f"- The cleaning run removed {dropped_rows:,} rows, matching the exact duplicate-removal policy rather than arbitrary row deletion.")
print(f"- Unsafe column names fell from {quality.loc['raw', 'unsafe_column_names']} to {quality.loc['cleaned', 'unsafe_column_names']}, making the cleaned schema easier to use programmatically.")
print(f"- The six targeted format findings were all resolved: inconsistent rows fell from {before_inconsistent:,} to {after_inconsistent:,}.")
print(f"- The accepted generated cleaners changed {int(cleaners['changed_rows'].sum()):,} row values across {len(cleaners)} columns, with host-side verification preserving already-valid examples.")
print("- Residual cross-column, anomaly, or near-duplicate risks should still be discussed separately because the experiment measures controlled improvement, not perfect data quality.")


,metric,raw,cleaned,change
0,Rows,20102,20077,-25
1,Unsafe column names,8,1,-7
2,Normalized exact duplicate rows,67,42,-25
3,Targeted inconsistent rows,4392,0,-4392
4,Resolved targeted columns,0,6,6


README bullets:
- The cleaning run removed 25 rows, matching the exact duplicate-removal policy rather than arbitrary row deletion.
- Unsafe column names fell from 8 to 1, making the cleaned schema easier to use programmatically.
- The six targeted format findings were all resolved: inconsistent rows fell from 4,392 to 0.
- The accepted generated cleaners changed 4,955 row values across 6 columns, with host-side verification preserving already-valid examples.
- Residual cross-column, anomaly, or near-duplicate risks should still be discussed separately because the experiment measures controlled improvement, not perfect data quality.


## Short Interpretation

Use this final cell as a compact result statement for the experimental section. It reports whether the cleaned output improved the targeted, measurable quality dimensions without claiming that all residual risks disappeared.

In [13]:
before_inconsistent = int(verification["before_inconsistent_rows"].sum()) if not verification.empty else 0
after_inconsistent = int(verification["after_inconsistent_rows"].sum()) if not verification.empty else 0
resolved = int((verification["status"] == "resolved").sum()) if not verification.empty else 0
dropped_rows = int(raw.shape[0] - cleaned.shape[0])

print(f"Dataset: {DATASET}")
print(f"Rows: raw={raw.shape[0]:,}, cleaned={cleaned.shape[0]:,}, dropped={dropped_rows:,}")
print(f"Unsafe column names: raw={quality.loc['raw', 'unsafe_column_names']}, cleaned={quality.loc['cleaned', 'unsafe_column_names']}")
print(f"Normalized exact duplicate rows: raw={quality.loc['raw', 'normalized_exact_duplicate_rows']}, cleaned={quality.loc['cleaned', 'normalized_exact_duplicate_rows']}")
print(f"Targeted format inconsistencies: raw={before_inconsistent:,}, cleaned={after_inconsistent:,}, resolved_columns={resolved}/{len(verification)}")
print(f"Cleaning summary: {final_report.get('cleaning_summary')}")
print(f"Verification summary: {final_report.get('verification_summary')}")

Dataset: attivazioniCessazioni
Rows: raw=20,102, cleaned=20,077, dropped=25
Unsafe column names: raw=8, cleaned=1
Normalized exact duplicate rows: raw=67, cleaned=42
Targeted format inconsistencies: raw=4,392, cleaned=0, resolved_columns=6/6
Cleaning summary: Applied 6 format cleaners, replaced 1517 placeholder values, renamed 7 columns, cast dtypes, lowercased 73123 string values, and dropped 25 exact duplicate row(s). Cleaned dataset saved to `/Users/mattia/Documents/GitHub/AgentsAI/Data/.cleaning_cache/attivazioniCessazioni/attivazioniCessazioni.cleaned.csv`.
Verification summary: 6 resolved (mese, anno, attivazioni, cessazioni, RATA→rata, aggregation-time→aggregation_time)
